In [10]:
from pathlib import Path
import pandas as pd

# Project directories

generated_dir = Path("../data/processed/generated")
evaluation_dir = Path("../data/evaluation")
project_dir = Path(
    "/Users/melaniequ/Voice Cloning and Fake Audio Detection/"
    "voice_cloning"
)

# Find all generated WAV files
audio_files = sorted(generated_dir.rglob("*.WAV_fake.wav"))

if not audio_files:
    print(f"No WAV files found in {generated_dir}")
else:
    manifest = pd.DataFrame({
        "generated_audio": [
            path
            for path in audio_files
        ],
        "reference_text": ["" for _ in audio_files],
        "target_speaker": ["" for _ in audio_files],
    })

    output_path = "../data/evaluation/vc_test_manifest.csv"
    manifest.to_csv(output_path, index=False)

    print(f"Found {len(audio_files)} WAV files.")
    print(f"Created manifest: {output_path}")
    print(manifest.head())

Found 740 WAV files.
Created manifest: ../data/evaluation/vc_test_manifest.csv
                                     generated_audio reference_text  \
0  ../data/processed/generated/FBCH0_SA2.WAV_fake...                  
1  ../data/processed/generated/FBCH0_SI1586.WAV_f...                  
2  ../data/processed/generated/FBCH0_SI956.WAV_fa...                  
3  ../data/processed/generated/FBCH0_SI959.WAV_fa...                  
4  ../data/processed/generated/FBCH0_SX146.WAV_fa...                  

  target_speaker  
0                 
1                 
2                 
3                 
4                 


In [13]:
from pathlib import Path
import pandas as pd
import re

# Update this if your original TIMIT dataset is stored elsewhere.

generated_dir = Path("../data/processed/generated")
evaluation_dir = "../data/evaluation"

project_dir = Path('../data/raw/timit')

# Index all TIMIT transcript files by their filename stem.
# Example: FBCH0_SA2.TXT -> FBCH0_SA2
transcript_index = {}

for txt_file in project_dir.rglob("*.TXT"):
    transcript_index[txt_file.stem.upper()] = txt_file

def read_timit_transcript(txt_path):
    """Read a TIMIT transcript, removing its leading utterance ID."""
    content = txt_path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()

    # Typical format: "SA2 She had your dark suit..."
    parts = content.split(maxsplit=1)

    if len(parts) == 2:
        return parts[1].strip()

    return ""

audio_files = sorted(
    p for p in generated_dir.rglob("*")
    if p.is_file() and p.suffix.lower() == ".wav"
)

rows = []

for audio_path in audio_files:
    # Example: FBCH0_SA2.WAV_fake.wav
    # Remove the final "_fake" from the filename stem.
    original_stem = re.sub(
        r"_fake$",
        "",
        audio_path.stem,
        flags=re.IGNORECASE
    )

    # Result: FBCH0_SA2.WAV -> FBCH0_SA2
    original_stem = re.sub(
        r"\.wav$",
        "",
        original_stem,
        flags=re.IGNORECASE
    )

    # Assuming the filename follows SPEAKER_UTTERANCE.
    speaker_id = original_stem.split("_")[0]

    transcript_path = transcript_index.get(
        original_stem.upper()
    )

    reference_text = (
        read_timit_transcript(transcript_path)
        if transcript_path
        else ""
    )

    rows.append({
        "generated_audio": str(
            audio_path
        ),
        "reference_text": reference_text,
        "target_speaker": speaker_id
    })

manifest = pd.DataFrame(rows)

output_path = "../data/evaluation/vc_test_manifest.csv"
manifest.to_csv(output_path, index=False)

print("Audio files found:", len(audio_files))
print("Transcripts matched:", manifest["reference_text"].ne("").sum())
print("Missing transcripts:", manifest["reference_text"].eq("").sum())
print("Manifest saved to:", output_path)
print(manifest.head(10))

Audio files found: 741
Transcripts matched: 0
Missing transcripts: 741
Manifest saved to: ../data/evaluation/vc_test_manifest.csv
                                     generated_audio reference_text  \
0  ../data/processed/generated/FBCH0_SA2.WAV_fake...                  
1  ../data/processed/generated/FBCH0_SI1586.WAV_f...                  
2  ../data/processed/generated/FBCH0_SI956.WAV_fa...                  
3  ../data/processed/generated/FBCH0_SI959.WAV_fa...                  
4  ../data/processed/generated/FBCH0_SX146.WAV_fa...                  
5  ../data/processed/generated/FBCH0_SX236.WAV_fa...                  
6  ../data/processed/generated/FBCH0_SX326.WAV_fa...                  
7  ../data/processed/generated/FBCH0_SX416.WAV_fa...                  
8  ../data/processed/generated/FBCH0_SX56.WAV_fak...                  
9  ../data/processed/generated/FCEG0_SA2.WAV_fake...                  

  target_speaker  
0          FBCH0  
1          FBCH0  
2          FBCH0  
3          F

In [14]:
from pathlib import Path

timit_dir = Path("../data/raw/timit").resolve()

print("TIMIT directory exists:", timit_dir.exists())

for path in list(timit_dir.rglob("*"))[:40]:
    print(path.relative_to(timit_dir))

TIMIT directory exists: True
test
train
.ipynb_checkpoints
test/DR4
test/DR3
test/DR2
test/DR5
test/DR7
test/DR6
test/DR1
test/DR8
test/.ipynb_checkpoints
test/DR4/MGMM0
test/DR4/MTLS0
test/DR4/FNMR0
test/DR4/FRNG0
test/DR4/MPLB0
test/DR4/FLKD0
test/DR4/FREW0
test/DR4/MJRF0
test/DR4/FCRH0
test/DR4/MLLL0
test/DR4/FADG0
test/DR4/MKCL0
test/DR4/FJMG0
test/DR4/MTEB0
test/DR4/FMCM0
test/DR4/MDLS0
test/DR4/MDRM0
test/DR4/FDMS0
test/DR4/FEDW0
test/DR4/FGJD0
test/DR4/MLJB0
test/DR4/MROA0
test/DR4/FJLM0
test/DR4/FLBW0
test/DR4/MPWM0
test/DR4/FCFT0
test/DR4/MBNS0
test/DR4/.ipynb_checkpoints


In [15]:
from pathlib import Path

timit_dir = Path("../data/raw/timit").resolve()

# Find transcript files regardless of extension capitalization
txt_files = [
    p for p in timit_dir.rglob("*")
    if p.is_file() and p.suffix.lower() == ".txt"
]

print("Total transcript files:", len(txt_files))

for path in txt_files[:15]:
    print(path.relative_to(timit_dir))
    print("Content:", path.read_text(
        encoding="utf-8", errors="replace"
    ).strip()[:150])
    print("-" * 50)

Total transcript files: 5670
test/DR4/MGMM0/SX139.TXT
Content: 0 44750 The bungalow was pleasantly situated near the shore.
--------------------------------------------------
test/DR4/MGMM0/SX229.TXT
Content: 0 28673 Are you looking for employment?
--------------------------------------------------
test/DR4/MGMM0/SA1.TXT
Content: 0 49051 She had your dark suit in greasy wash water all year.
--------------------------------------------------
test/DR4/MGMM0/SX49.TXT
Content: 0 60417 At twilight on the twelfth day we'll have Chablis.
--------------------------------------------------
test/DR4/MGMM0/SA2.TXT
Content: 0 45160 Don't ask me to carry an oily rag like that.
--------------------------------------------------
test/DR4/MGMM0/SX409.TXT
Content: 0 55195 Eating spinach nightly increases strength miraculously.
--------------------------------------------------
test/DR4/MGMM0/SI1759.TXT
Content: 0 45057 Got a heck of a buy on this, dirt cheap.
-------------------------------------------

In [16]:
from pathlib import Path
import pandas as pd
import re

generated_dir = Path("../data/processed/generated").resolve()
timit_dir = Path("../data/raw/timit").resolve()
evaluation_dir = Path("../data/evaluation").resolve()
evaluation_dir.mkdir(parents=True, exist_ok=True)

# Index transcripts by (speaker ID, utterance ID)
transcript_index = {}

for path in timit_dir.rglob("*"):
    if path.is_file() and path.suffix.lower() == ".txt":
        # Expected nested structure: .../speaker/SA2.TXT
        speaker_id = path.parent.name.upper()
        utterance_id = path.stem.upper()

        transcript_index[(speaker_id, utterance_id)] = path

def read_transcript(path):
    content = path.read_text(
        encoding="utf-8", errors="replace"
    ).strip()

    # TIMIT format: start_time end_time transcript
    parts = content.split(maxsplit=2)
    return parts[2].strip() if len(parts) >= 3 else ""

audio_files = sorted(
    p for p in generated_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() == ".wav"
    and "_fake" in p.stem.lower()
)

rows = []

for audio_path in audio_files:
    # FBCH0_SA2.WAV_fake.wav -> FBCH0_SA2
    original_stem = re.sub(
        r"_fake$", "", audio_path.stem, flags=re.IGNORECASE
    )
    original_stem = re.sub(
        r"\.wav$", "", original_stem, flags=re.IGNORECASE
    )

    if "_" not in original_stem:
        rows.append({
            "generated_audio": str(audio_path),
            "reference_text": "",
            "target_speaker": "",
            "transcript_found": False,
            "transcript_path": ""
        })
        continue

    speaker_id, utterance_id = original_stem.split("_", 1)

    transcript_path = transcript_index.get(
        (speaker_id.upper(), utterance_id.upper())
    )

    rows.append({
        "generated_audio": str(audio_path),
        "reference_text": (
            read_transcript(transcript_path)
            if transcript_path else ""
        ),
        "target_speaker": speaker_id.upper(),
        "transcript_found": transcript_path is not None,
        "transcript_path": (
            str(transcript_path) if transcript_path else ""
        )
    })

manifest = pd.DataFrame(rows)
output_path = evaluation_dir / "vc_test_manifest.csv"
manifest.to_csv(output_path, index=False)

print("Generated audio files:", len(audio_files))
print("Transcripts indexed:", len(transcript_index))
print("Transcripts matched:", manifest["transcript_found"].sum())
print("Missing transcripts:", (~manifest["transcript_found"]).sum())
print("Manifest saved to:", output_path)

display(manifest.head(10))

Generated audio files: 740
Transcripts indexed: 5670
Transcripts matched: 740
Missing transcripts: 0
Manifest saved to: /Users/melaniequ/Voice Cloning and Fake Audio Detection/data/evaluation/vc_test_manifest.csv


,generated_audio,reference_text,target_speaker,transcript_found,transcript_path
0,/Users/melaniequ/Voice Cloning and Fake Audio ...,Don't ask me to carry an oily rag like that.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
1,/Users/melaniequ/Voice Cloning and Fake Audio ...,"Your voice is delightful, he approved with a w...",FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
2,/Users/melaniequ/Voice Cloning and Fake Audio ...,Turbulent tides rose as much as fifty feet.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
3,/Users/melaniequ/Voice Cloning and Fake Audio ...,Sometimes soldiers wrote letters while bullets...,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
4,/Users/melaniequ/Voice Cloning and Fake Audio ...,Cyclical programs will never compile.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
5,/Users/melaniequ/Voice Cloning and Fake Audio ...,Clear pronunciation is appreciated.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
6,/Users/melaniequ/Voice Cloning and Fake Audio ...,Steve collects rare and novel coins.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
7,/Users/melaniequ/Voice Cloning and Fake Audio ...,A huge power outage rarely occurs.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
8,/Users/melaniequ/Voice Cloning and Fake Audio ...,Academic aptitude guarantees your diploma.,FBCH0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
9,/Users/melaniequ/Voice Cloning and Fake Audio ...,Don't ask me to carry an oily rag like that.,FCEG0,True,/Users/melaniequ/Voice Cloning and Fake Audio ...
